# 🔬 Clase 3 — Implementación de modelos de regresión lineal con statsmodels
## Unidad: Correlación y modelamiento

**Situación:** El área de estrategia quiere saber si el tiempo de navegación impacta el gasto final. Tu misión es construir el modelo, **leer la salida completa de `modelo.summary()`** y comunicar los hallazgos de forma ejecutiva.

**Preguntas clave:**
- ¿Qué significan cada uno de los valores que entrega `summary()`?
- ¿Cómo sé si el modelo es estadísticamente válido?
- ¿Cómo comunico los hallazgos a un público no técnico?

**Objetivos:**
- Preparar datos para `statsmodels` (`sm.add_constant`)
- Interpretar la **tabla completa de `summary()`**: coef, std err, t, P>|t|, IC 95%
- Extraer parámetros clave con `modelo.params`, `modelo.pvalues`, `modelo.conf_int()`
- Generar un **reporte ejecutivo automatizado**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')
print(f'statsmodels {sm.__version__}')

---
## PARTE 1 — Preparación de datos

### 1.1 Cargar el dataset

In [ ]:
# Código exacto de la presentación
url = 'Material_de_apoyo_Tiempo_de_navegacion.csv'
df = pd.read_csv(url)

print(df.head())

In [ ]:
print(df.info())
print()
print(df.describe().round(2))

### 1.2 `sm.add_constant()` — ¿por qué es obligatorio en statsmodels?

In [ ]:
# Código exacto de la presentación
X = sm.add_constant(df['tiempo_navegacion'])   # Variable explicativa + intercepto
Y = df['gasto']                                 # Variable dependiente

print('=== X sin add_constant: solo la variable explicativa ===')
print(df['tiempo_navegacion'].head(3).to_frame())
print()
print('=== X CON add_constant: columna "const" = 1 agregada ===')
print(X.head(3))
print()
print('¿Por qué? statsmodels no agrega el intercepto automáticamente.')
print('Sin sm.add_constant() el modelo fuerza la recta a pasar por el origen (β₀ = 0).')

In [ ]:
# Demostración: modelo SIN intercepto vs modelo CON intercepto
modelo_sin = sm.OLS(Y, df[['tiempo_navegacion']]).fit()   # sin const
modelo_con = sm.OLS(Y, X).fit()                           # con const

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Efecto de sm.add_constant() — sin vs con intercepto', fontweight='bold')

x_r = np.linspace(df['tiempo_navegacion'].min(), df['tiempo_navegacion'].max(), 100)
for ax, modelo, titulo, color in zip(
    axes,
    [modelo_sin, modelo_con],
    ['SIN add_constant (forzado a pasar por origen)',
     'CON add_constant (intercepto libre)'],
    ['#FC4E4E', '#2E75B6']
):
    ax.scatter(df['tiempo_navegacion'], Y, color='#A5A5A5', s=25, alpha=0.5)
    params = modelo.params
    if 'const' in params.index:
        recta = params['const'] + params['tiempo_navegacion'] * x_r
        etiq  = f"ŷ = {params['const']:,.0f} + {params['tiempo_navegacion']:.0f}·X"
    else:
        recta = params['tiempo_navegacion'] * x_r
        etiq  = f"ŷ = {params['tiempo_navegacion']:.0f}·X  (sin intercepto)"
    ax.plot(x_r, recta, color=color, linewidth=2.5, label=etiq)
    ax.set_title(titulo, fontsize=9)
    ax.set_xlabel('Tiempo (min)')
    ax.set_ylabel('Gasto ($)')
    ax.legend(fontsize=8)
    ax.text(0.05, 0.92, f'R²={modelo.rsquared:.3f}',
            transform=ax.transAxes, fontsize=10, color=color, fontweight='bold')

plt.tight_layout()
plt.show()

### ✏️ Ejercicio 1:

In [ ]:
# ✏️ ¿Qué columna agrega sm.add_constant() y qué valor tiene?
r_const = ""

# ✏️ ¿Qué problema ocurre si omites sm.add_constant()?
r_sin_const = ""

# ✏️ ¿En qué caso tendría sentido forzar un modelo sin intercepto?
r_sin_intercepto = ""

print(f'Columna const: {r_const}')
print(f'Sin add_constant: {r_sin_const}')
print(f'Sin intercepto tiene sentido: {r_sin_intercepto}')

---
## PARTE 2 — Ajuste del modelo y lectura completa de `summary()`

### 2.1 Ajustar el modelo

In [ ]:
# Código exacto de la presentación
modelo = sm.OLS(Y, X).fit()
print(modelo.summary())

### 2.2 Anatomía del `summary()` — sección por sección

In [ ]:
print('=' * 62)
print('BLOQUE 1 — Métricas globales del modelo')
print('=' * 62)
print(f'  R²:              {modelo.rsquared:.4f}  → % variabilidad de Y explicada por X')
print(f'  R² ajustado:     {modelo.rsquared_adj:.4f}  → penaliza si se añaden variables')
print(f'  F-statistic:     {modelo.fvalue:.4f}  → ¿el modelo en su conjunto es significativo?')
print(f'  Prob(F):         {modelo.f_pvalue:.6f}  → si < 0.05, el modelo es significativo')
print(f'  N observaciones: {int(modelo.nobs)}')
print(f'  AIC:             {modelo.aic:.2f}   (menor = mejor ajuste entre modelos)')
print(f'  BIC:             {modelo.bic:.2f}   (menor = mejor ajuste penalizando complejidad)')

In [ ]:
print('=' * 62)
print('BLOQUE 2 — Tabla de coeficientes')
print('=' * 62)

tabla_coef = pd.DataFrame({
    'coef':    modelo.params,
    'std err': modelo.bse,
    't':       modelo.tvalues,
    'P>|t|':   modelo.pvalues,
    'IC inf (2.5%)':  modelo.conf_int()[0],
    'IC sup (97.5%)': modelo.conf_int()[1]
}).round(4)
print(tabla_coef)

print()
print('Columna por columna:')
print('  coef     → valor estimado del parámetro (β₀ o β₁)')
print('  std err  → incertidumbre en la estimación del coeficiente')
print('  t        → estadístico t = coef / std err')
print('  P>|t|    → valor p: si < 0.05, el coeficiente es estadísticamente significativo')
print('  [0.025 0.975] → intervalo de confianza al 95%: rango plausible del coeficiente')

In [ ]:
b0 = modelo.params['const']
b1 = modelo.params['tiempo_navegacion']
p0 = modelo.pvalues['const']
p1 = modelo.pvalues['tiempo_navegacion']
ic = modelo.conf_int()

print('=' * 62)
print('INTERPRETACIÓN CONTEXTUALIZADA')
print('=' * 62)
print(f'β₀ (intercepto):')
print(f'  Valor: ${b0:,.0f}')
print(f'  p-value: {p0:.4f} → {"significativo ✅" if p0 < 0.05 else "NO significativo ⚠️"}')
print(f'  IC 95%: [${ic.loc["const",0]:,.0f} — ${ic.loc["const",1]:,.0f}]')
print(f'  Interpretación: gasto base esperado cuando tiempo = 0')
print()
print(f'β₁ (pendiente tiempo_navegacion):')
print(f'  Valor: ${b1:,.0f} por minuto')
print(f'  p-value: {p1:.6f} → {"significativo ✅" if p1 < 0.05 else "NO significativo ⚠️"}')
print(f'  IC 95%: [${ic.loc["tiempo_navegacion",0]:,.0f} — ${ic.loc["tiempo_navegacion",1]:,.0f}]')
print(f'  Interpretación: por cada minuto adicional de navegación, el gasto aumenta en ${b1:,.0f}')

### 2.3 Visualizar los intervalos de confianza

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Modelo OLS — Recta de regresión e intervalos de confianza', fontweight='bold')

# Recta con banda de confianza (regplot la calcula automáticamente)
sns.regplot(data=df, x='tiempo_navegacion', y='gasto',
            ax=axes[0], ci=95, color='#2E75B6',
            scatter_kws={'alpha': 0.4, 's': 25},
            line_kws={'linewidth': 2.5})
axes[0].set_title(f'ŷ = ${b0:,.0f} + ${b1:,.0f}·X  |  R²={modelo.rsquared:.3f}')
axes[0].set_xlabel('Tiempo de navegación (min)')
axes[0].set_ylabel('Gasto ($)')

# Gráfico de coeficientes con IC
coefs = tabla_coef.drop(columns=['std err','t'])
coefs_plot = coefs[['coef','IC inf (2.5%)','IC sup (97.5%)']]
y_pos = [1, 0]
for i, (idx, row) in enumerate(coefs_plot.iterrows()):
    axes[1].barh(i, row['coef'], xerr=[[row['coef']-row['IC inf (2.5%)']],
                                        [row['IC sup (97.5%)']-row['coef']]],
                 color='#2E75B6', capsize=6, height=0.4,
                 error_kw={'ecolor': '#1F4E79', 'lw': 2})
    axes[1].text(row['IC sup (97.5%)']*1.02, i, f'{row["coef"]:,.1f}', va='center', fontsize=9)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['tiempo_navegacion (β₁)', 'const (β₀)'])
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Coeficientes con IC 95%')
axes[1].set_xlabel('Valor del coeficiente')

plt.tight_layout()
plt.show()

### ✏️ Ejercicio 2 — Interpreta la salida:

In [ ]:
# ✏️ ¿Qué indica el p-value de β₁? ¿Es el modelo estadísticamente significativo?
r_pvalue = ""

# ✏️ ¿Qué te dice el intervalo de confianza de β₁ sobre la precisión del modelo?
ic_b1_rango = ic.loc['tiempo_navegacion', 1] - ic.loc['tiempo_navegacion', 0]
print(f'IC β₁: [{ic.loc["tiempo_navegacion",0]:,.0f} — {ic.loc["tiempo_navegacion",1]:,.0f}]')
print(f'Rango IC: ${ic_b1_rango:,.0f}')
r_ic = ""

# ✏️ Si el R² es 0.72, ¿qué significa el 28% restante?
r_28pct = ""

# ✏️ Un ejecutivo te pregunta: "¿Este modelo es confiable para tomar decisiones?"
# ¿Qué le dirías basándote en los valores obtenidos?
r_ejecutivo = ""

print(f'\np-value β₁: {r_pvalue}')
print(f'IC β₁:      {r_ic}')
print(f'28% resto:  {r_28pct}')
print(f'Al ejecutivo: {r_ejecutivo}')

---
## PARTE 3 — Extracción de elementos clave del modelo

### 3.1 Acceso programático a los resultados

In [ ]:
# Código exacto de la presentación
print('Intercepto:', modelo.params['const'])
print('Pendiente: ', modelo.params['tiempo_navegacion'])
print('R²:        ', modelo.rsquared)
print('Valor p:   ', modelo.pvalues['tiempo_navegacion'])

In [ ]:
# Extracción completa — presentación
print('=== modelo.params ===')
print(modelo.params)

print()
print('=== modelo.pvalues ===')
print(modelo.pvalues)

print()
print('=== modelo.conf_int() ===')
print(modelo.conf_int())

### 3.2 Generar un reporte ejecutivo automatizado

In [ ]:
# Reporte listo para compartir con la gerencia
def reporte_ejecutivo(modelo, var_x, var_y, unidad_x='min', unidad_y='$'):
    b0 = modelo.params['const']
    b1 = modelo.params[var_x]
    p1 = modelo.pvalues[var_x]
    r2 = modelo.rsquared
    ic = modelo.conf_int()
    n  = int(modelo.nobs)
    sig = 'estadísticamente significativo (p < 0.05) ✅' if p1 < 0.05 else 'NO significativo ⚠️'

    print('═' * 65)
    print('  REPORTE EJECUTIVO — MODELO DE REGRESIÓN LINEAL')
    print('═' * 65)
    print(f'  Variable predictora: {var_x} ({unidad_x})')
    print(f'  Variable respuesta:  {var_y} ({unidad_y})')
    print(f'  N observaciones:     {n}')
    print()
    print('  ECUACIÓN DEL MODELO')
    print(f'  ŷ = {b0:,.0f} + {b1:,.0f} × {var_x}')
    print()
    print('  INTERPRETACIÓN')
    print(f'  • Gasto base estimado (tiempo = 0):     {unidad_y}{b0:,.0f}')
    print(f'  • Incremento por 1 {unidad_x} de navegación: +{unidad_y}{b1:,.0f}')
    print(f'  • IC 95% pendiente: [{unidad_y}{ic.loc[var_x,0]:,.0f} — {unidad_y}{ic.loc[var_x,1]:,.0f}]')
    print()
    print('  CALIDAD DEL MODELO')
    print(f'  • R² = {r2:.4f} → el modelo explica el {r2*100:.1f}% de la variabilidad del gasto')
    print(f'  • El predictor es {sig}')
    print(f'  • p-value = {p1:.6f}')
    print()
    print('  EJEMPLO DE PREDICCIÓN')
    for t in [5, 10, 20]:
        pred = b0 + b1 * t
        print(f'  • Cliente que navega {t:>2} {unidad_x}: gasto estimado = {unidad_y}{pred:,.0f}')
    print('═' * 65)

reporte_ejecutivo(modelo, 'tiempo_navegacion', 'gasto')

### 3.3 Diagnóstico visual completo

In [ ]:
pred = modelo.predict(X)
residuos = Y - pred

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Diagnóstico completo del modelo OLS', fontweight='bold', fontsize=13)

# 1. Scatter + recta
axes[0,0].scatter(df['tiempo_navegacion'], Y, color='#2E75B6', alpha=0.5, s=30)
x_r = np.linspace(df['tiempo_navegacion'].min(), df['tiempo_navegacion'].max(), 100)
axes[0,0].plot(x_r, b0 + b1 * x_r, color='#ED7D31', linewidth=2.5)
axes[0,0].set_title(f'Recta OLS  |  R²={modelo.rsquared:.3f}')
axes[0,0].set_xlabel('Tiempo (min)')
axes[0,0].set_ylabel('Gasto ($)')

# 2. Residuos vs predichos
axes[0,1].scatter(pred, residuos, color='#7030A0', alpha=0.6, s=30)
axes[0,1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_title('Residuos vs Valores predichos')
axes[0,1].set_xlabel('ŷ')
axes[0,1].set_ylabel('Residuo (Y - ŷ)')

# 3. QQ-plot de residuos (normalidad)
from scipy import stats
(osm, osr), (slope, intercept, r) = stats.probplot(residuos)
axes[1,0].scatter(osm, osr, color='#70AD47', s=25, alpha=0.7)
axes[1,0].plot(osm, slope*np.array(osm)+intercept, color='red', linewidth=2)
axes[1,0].set_title('QQ-plot — Normalidad de residuos')
axes[1,0].set_xlabel('Cuantiles teóricos')
axes[1,0].set_ylabel('Cuantiles observados')

# 4. Histograma de residuos
sns.histplot(residuos, bins=15, kde=True, ax=axes[1,1], color='#BDD7EE', edgecolor='white')
axes[1,1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1,1].set_title('Distribución de residuos (debe ser ~normal centrada en 0)')
axes[1,1].set_xlabel('Residuo')

plt.tight_layout()
plt.show()

print(f'Media de residuos:  {residuos.mean():.4f}  ← debe ser ≈ 0')
print(f'Std de residuos:    {residuos.std():.2f}')

---
## PARTE 4 — Actividad guiada: Informe para el área de ventas

### 4.1 Reproducir el flujo completo paso a paso

In [ ]:
# Pasos exactos de la presentación — actividad guiada

# Paso 1: Cargar
import pandas as pd
df_g = pd.read_csv('Material_de_apoyo_Tiempo_de_navegacion.csv')

# Paso 2: Ajustar modelo
import statsmodels.api as sm
X_g = sm.add_constant(df_g['tiempo_navegacion'])
Y_g = df_g['gasto']
modelo_g = sm.OLS(Y_g, X_g).fit()

# Paso 3: Revisar salida
print(modelo_g.summary())

In [ ]:
# Paso 4: Extraer información clave — código exacto de la presentación
print(modelo_g.params)
print(modelo_g.pvalues)
print(modelo_g.conf_int())

### ✏️ Tabla de análisis (como en la presentación):

In [ ]:
# Completa esta tabla (similar a la de la Slide 24)
tabla_analisis = pd.DataFrame({
    'Elemento a analizar': [
        'Intercepto (const)',
        'Coeficiente (pendiente)',
        'Valor de R²',
        'Valor p del predictor',
        '¿Es un modelo significativo?',
        '¿Es útil para predecir el gasto?'
    ],
    'Valor obtenido': [
        f"{modelo_g.params['const']:,.2f}",
        f"{modelo_g.params['tiempo_navegacion']:,.2f}",
        f"{modelo_g.rsquared:.4f}",
        f"{modelo_g.pvalues['tiempo_navegacion']:.6f}",
        'Sí ✅' if modelo_g.pvalues['tiempo_navegacion'] < 0.05 else 'No ⚠️',
        ''
    ],
    'Interpretación breve': [
        'Gasto esperado cuando tiempo = 0',
        'Gasto adicional por cada minuto más de navegación',
        f'Explica el {modelo_g.rsquared*100:.1f}% de la variabilidad',
        'p < 0.05 → relación estadísticamente significativa',
        '',
        '✏️ Completa aquí tu respuesta'
    ]
})
print(tabla_analisis.to_string(index=False))

---
## PARTE 5 — Actividad autónoma: productos vistos vs gasto total

### 5.1 Crear el DataFrame exacto de la presentación

In [ ]:
# Código exacto de la presentación
import pandas as pd
data = {
    'cliente':         ['A1','A2','A3','A4','A5','A6','A7','A8','A9','A10'],
    'productos_vistos':[3,    7,   4,   5,   2,   8,   6,   4,   9,   5],
    'gasto_total':     [950, 1450,1100,1200, 900,1600,1300,1150,1700,1250]
}
df_auto = pd.DataFrame(data)
print(df_auto)
print()
print(df_auto.describe().round(2))

### 5.2 Exploración visual previa

In [ ]:
r_auto = df_auto['productos_vistos'].corr(df_auto['gasto_total'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Exploración previa — productos vistos vs gasto total', fontweight='bold')

sns.scatterplot(data=df_auto, x='productos_vistos', y='gasto_total',
                ax=axes[0], color='#2E75B6', s=100)
# Añadir etiquetas de cliente
for _, row in df_auto.iterrows():
    axes[0].text(row['productos_vistos']+0.1, row['gasto_total']+10,
                 row['cliente'], fontsize=8, color='gray')
axes[0].set_title(f'Dispersión (r = {r_auto:.3f})')
axes[0].set_xlabel('Productos vistos')
axes[0].set_ylabel('Gasto total ($)')

sns.regplot(data=df_auto, x='productos_vistos', y='gasto_total',
            ax=axes[1], ci=95, color='#70AD47',
            scatter_kws={'s': 80},
            line_kws={'linewidth': 2.5})
axes[1].set_title('Con línea de tendencia + IC 95%')
axes[1].set_xlabel('Productos vistos')
axes[1].set_ylabel('Gasto total ($)')

plt.tight_layout()
plt.show()

### 5.3 Implementar el modelo — código exacto de la presentación

In [ ]:
# Código exacto de la presentación
import statsmodels.api as sm
X_a = sm.add_constant(df_auto['productos_vistos'])
Y_a = df_auto['gasto_total']
modelo_auto = sm.OLS(Y_a, X_a).fit()

# Revisión completa — código exacto de la presentación
print(modelo_auto.summary())

In [ ]:
# Código exacto de la presentación
print('Coeficientes:\n', modelo_auto.params)
print('R²:', modelo_auto.rsquared)
print('Valores p:\n', modelo_auto.pvalues)
print('Intervalos de confianza:\n', modelo_auto.conf_int())

### 5.4 Completar la tabla de análisis (como en la presentación)

In [ ]:
# ✏️ Completa con tus propias interpretaciones:
tabla_auto = pd.DataFrame({
    'Elemento': [
        'Intercepto (const)',
        'Coeficiente (pendiente)',
        'Valor de R²',
        'Valor p del predictor',
        '¿Es un modelo significativo?',
        '¿Es útil para predecir el gasto?'
    ],
    'Valor': [
        f"{modelo_auto.params['const']:,.2f}",
        f"{modelo_auto.params['productos_vistos']:,.2f}",
        f"{modelo_auto.rsquared:.4f}",
        f"{modelo_auto.pvalues['productos_vistos']:.6f}",
        '',
        ''
    ],
    'Tu interpretación': ['', '', '', '', '', '']
})
print(tabla_auto.to_string(index=False))

### ✏️ Preguntas de reflexión finales:

In [ ]:
b0_a = modelo_auto.params['const']
b1_a = modelo_auto.params['productos_vistos']

# ✏️ 1. ¿Qué tan preciso es el modelo con solo 10 observaciones?
c1 = ""

# ✏️ 2. ¿Qué diferencias observas respecto al modelo de tiempo de navegación?
c2 = ""

# ✏️ 3. ¿Cuál de las métricas es más útil para comunicar el desempeño a la gerencia?
c3 = ""

# ✏️ 4. Aplica el reporte ejecutivo al modelo autónomo:
reporte_ejecutivo(modelo_auto, 'productos_vistos', 'gasto_total',
                  unidad_x='productos', unidad_y='$')

print(f'\n1. Precisión con n=10: {c1}')
print(f'2. Diferencias:        {c2}')
print(f'3. Métrica para gerencia: {c3}')

---
## 📋 Resumen de atributos del modelo y la salida `summary()`

| Atributo | Código | Descripción |
|----------|--------|-------------|
| Coeficientes | `modelo.params` | β₀ y β₁ |
| Errores estándar | `modelo.bse` | Incertidumbre de cada coeficiente |
| Estadístico t | `modelo.tvalues` | coef / std err |
| P-values | `modelo.pvalues` | Significancia estadística |
| IC 95% | `modelo.conf_int()` | Rango plausible del coeficiente |
| R² | `modelo.rsquared` | Capacidad explicativa global |
| R² ajustado | `modelo.rsquared_adj` | Penaliza variables innecesarias |
| F-statistic | `modelo.fvalue` | Significancia global del modelo |
| p(F) | `modelo.f_pvalue` | p-value del estadístico F |
| AIC / BIC | `modelo.aic / .bic` | Comparar entre modelos |
| N observaciones | `modelo.nobs` | Tamaño de la muestra |
| Predicciones | `modelo.predict(X)` | Valores ŷ |
| Resumen completo | `modelo.summary()` | Todo lo anterior en una tabla |

**Checklist de validación del modelo:**

| Check | Qué verificar | Señal de alerta |
|-------|--------------|----------------|
| ✅ Intercepto | `modelo.params['const']` tiene sentido | X=0 no tiene interpretación práctica |
| ✅ Significancia β₁ | `pvalues['X'] < 0.05` | p > 0.05 → predictor no significativo |
| ✅ R² | > 0.5 para aplicaciones reales | R² alto pero p > 0.05 |
| ✅ Residuos | Distribuidos alrededor de 0 | Patrón sistemático en residuos |
| ✅ IC β₁ | No cruza el 0 | IC incluye 0 → coeficiente no confiable |

> 💡 **Regla de oro:** Si `IC 95%` de β₁ incluye el 0, el efecto no está bien establecido, aunque el valor del coeficiente parezca grande.

> 💡 **Automatiza el reporte:** Extrae los parámetros con `modelo.params`, `modelo.pvalues` y `modelo.conf_int()` para generar reportes ejecutivos reproducibles sin copiar y pegar manualmente.